In [1]:
%pip install -U mcp langchain langchain-mcp-adapters langchain-google-genai uv

  Using cached mcp-2.1.1-py3-none-any.whl.metadata (7.8 kB)
Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys

print(sys.executable)

C:\Users\Argit\anaconda3\envs\agenticai\python.exe


In [3]:
import shutil

print("UV :", shutil.which("uv"))
print("UVX:", shutil.which("uvx"))

UV : C:\Users\Argit\anaconda3\envs\agenticai\Scripts\uv.EXE
UVX: C:\Users\Argit\anaconda3\envs\agenticai\Scripts\uvx.EXE


In [4]:
import os
import sqlite3

folder_path = r"G:\My Drive\HOPE AI - AI TAMIL\MCP"
database_path = os.path.join(folder_path, "database.db")

# Create folder if it does not exist
os.makedirs(folder_path, exist_ok=True)

# Create SQLite database if it does not exist
connection = sqlite3.connect(database_path)
connection.close()

print("Database created successfully")
print("Database exists:", os.path.exists(database_path))
print("Database path:", database_path)

Database created successfully
Database exists: True
Database path: G:\My Drive\HOPE AI - AI TAMIL\MCP\database.db


In [5]:
import sqlite3

database_path = r"G:\My Drive\HOPE AI - AI TAMIL\MCP\database.db"

connection = sqlite3.connect(database_path)

cursor = connection.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS Employee
(
    ID INTEGER PRIMARY KEY,
    Name TEXT,
    Department TEXT,
    Salary REAL
)
""")

connection.commit()
connection.close()

print("Employee table created successfully")

Employee table created successfully


In [ ]:
import asyncio
import os
import sys
import shutil
import sqlite3
from getpass import getpass

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from langchain_mcp_adapters.tools import load_mcp_tools
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI


# ============================================================
# GOOGLE API KEY
# ============================================================

google_api_key = getpass("Enter your Google API Key: ").strip()

if not google_api_key:
    raise ValueError("Google API Key is required.")

os.environ["GOOGLE_API_KEY"] = google_api_key

print("Google API Key loaded successfully.")


# ============================================================
# DATABASE FOLDER
# ============================================================

folder_path = r"G:\My Drive\HOPE AI - AI TAMIL\MCP"

database_path = os.path.join(
    folder_path,
    "database.db"
)


# ============================================================
# CREATE FOLDER IF REQUIRED
# ============================================================

os.makedirs(
    folder_path,
    exist_ok=True
)


# ============================================================
# CREATE SQLITE DATABASE IF REQUIRED
# ============================================================

if not os.path.exists(database_path):

    connection = sqlite3.connect(database_path)

    connection.close()

    print("SQLite Database created successfully.")

else:

    print("SQLite Database already exists.")


print("Database Path:", database_path)


# ============================================================
# FIND UVX
# ============================================================

uvx_path = shutil.which("uvx")


if uvx_path is None:

    possible_uvx_path = os.path.join(
        os.path.dirname(sys.executable),
        "Scripts",
        "uvx.exe"
    )

    if os.path.exists(possible_uvx_path):
        uvx_path = possible_uvx_path


if uvx_path is None:

    raise FileNotFoundError(
        "uvx not found. Run `%pip install uv` and restart Jupyter."
    )


print("UVX Path:", uvx_path)


# ============================================================
# MCP ERROR LOG
# ============================================================

mcp_log_path = os.path.join(
    folder_path,
    "mcp_server_error.log"
)

print("MCP Log Path:", mcp_log_path)


# ============================================================
# INITIALIZE GEMINI MODEL
# ============================================================

model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    api_key=google_api_key,
    temperature=0
)


print("Gemini model initialized successfully.")


# ============================================================
# MCP SQLITE SERVER
#
# IMPORTANT:
#
# mcp-server-sqlite is an OLD MCP v1 server.
#
# Current MCP is v2.
#
# So force the SQLite server to run using:
#
#       mcp < 2
#
# The server runs in its own isolated UV environment.
# ============================================================

server_params = StdioServerParameters(

    command=uvx_path,

    args=[
        "--isolated",

        "--with",
        "mcp<2",

        "mcp-server-sqlite@2025.4.25",

        "--db-path",
        database_path
    ],

    env={
        **os.environ,
        "PYTHONIOENCODING": "utf-8"
    }
)


# ============================================================
# PROCESS QUERY
# ============================================================

async def process_query(agent, query):

    response = await agent.ainvoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": query
                }
            ]
        }
    )

    return response["messages"][-1].content


# ============================================================
# MAIN
# ============================================================

async def main():

    print("\n==========================================")
    print("Starting SQLite MCP Server")
    print("==========================================")

    print("SQLite MCP Server Version : 2025.4.25")
    print("SQLite Server MCP Version : < 2")
    print("Database                  :", database_path)

    print("\nStarting server...")


    # ========================================================
    # REAL ERROR LOG FILE
    #
    # Required because Jupyter sys.stderr causes
    # Windows fileno() problem.
    # ========================================================

    with open(
        mcp_log_path,
        "w",
        encoding="utf-8"
    ) as mcp_errlog:

        try:

            async with stdio_client(
                server_params,
                errlog=mcp_errlog
            ) as (read, write):

                async with ClientSession(
                    read,
                    write
                ) as session:


                    # ========================================
                    # INITIALIZE MCP
                    # ========================================

                    print("\nConnecting to MCP Server...")


                    await session.initialize()


                    print(
                        "MCP Server connected successfully."
                    )


                    # ========================================
                    # GET MCP TOOLS DIRECTLY
                    # ========================================

                    server_tools = await session.list_tools()


                    print("\nMCP Server Tools:")


                    for tool in server_tools.tools:

                        print(
                            " -",
                            tool.name
                        )


                    # ========================================
                    # LOAD MCP TOOLS INTO LANGCHAIN
                    # ========================================

                    tools = await load_mcp_tools(
                        session
                    )


                    print(
                        "\nLangChain loaded",
                        len(tools),
                        "MCP tools."
                    )


                    # ========================================
                    # CREATE LANGCHAIN AGENT
                    # ========================================

                    agent = create_agent(
                        model,
                        tools
                    )


                    print(
                        "LangChain Agent created successfully."
                    )


                    # ========================================
                    # START ASSISTANT
                    # ========================================

                    print("\n==========================================")
                    print("SQLite Database Assistant")
                    print("==========================================")
                    print("Type 'exit' to quit")
                    print("==========================================")


                    while True:


                        query = input(
                            "\nEnter your query: "
                        ).strip()


                        # ====================================
                        # EXIT
                        # ====================================

                        if query.lower() in [
                            "exit",
                            "quit"
                        ]:

                            print(
                                "\nSQLite Database Assistant closed."
                            )

                            break


                        # ====================================
                        # EMPTY QUERY
                        # ====================================

                        if not query:

                            continue


                        print("\nProcessing...")


                        try:


                            response = await process_query(
                                agent,
                                query
                            )


                            print("\nAnswer:")

                            print(response)


                        except Exception as query_error:


                            print("\nQuery Error:")

                            print(
                                type(query_error).__name__
                            )

                            print(
                                str(query_error)
                            )


        except BaseException as connection_error:


            print("\n==========================================")
            print("MCP SERVER CONNECTION FAILED")
            print("==========================================")


            print(
                "\nError Type:",
                type(connection_error).__name__
            )


            print(
                "\nError:"
            )

            print(
                str(connection_error)
            )


            # ================================================
            # PRINT MCP SERVER ERROR LOG
            # ================================================

            if os.path.exists(mcp_log_path):

                print("\n==========================================")
                print("MCP SERVER ERROR LOG")
                print("==========================================")

                try:

                    with open(
                        mcp_log_path,
                        "r",
                        encoding="utf-8"
                    ) as log_file:

                        log_content = (
                            log_file.read()
                        )


                    if log_content.strip():

                        print(log_content)

                    else:

                        print(
                            "MCP log file is empty."
                        )


                except Exception as log_error:

                    print(
                        "Unable to read MCP log:",
                        log_error
                    )


# ============================================================
# RUN IN JUPYTER
# ============================================================

await main()

Enter your Google API Key:  ········


Google API Key loaded successfully.
SQLite Database already exists.
Database Path: G:\My Drive\HOPE AI - AI TAMIL\MCP\database.db
UVX Path: C:\Users\Argit\anaconda3\envs\agenticai\Scripts\uvx.EXE
MCP Log Path: G:\My Drive\HOPE AI - AI TAMIL\MCP\mcp_server_error.log
Gemini model initialized successfully.

Starting SQLite MCP Server
SQLite MCP Server Version : 2025.4.25
SQLite Server MCP Version : < 2
Database                  : G:\My Drive\HOPE AI - AI TAMIL\MCP\database.db

Starting server...

Connecting to MCP Server...
MCP Server connected successfully.

MCP Server Tools:
 - read_query
 - write_query
 - create_table
 - list_tables
 - describe_table
 - append_insight

LangChain loaded 6 MCP tools.
LangChain Agent created successfully.

SQLite Database Assistant
Type 'exit' to quit



Enter your query:  list the all tables



Processing...

Answer:
[{'type': 'text', 'text': 'The tables in the database are: Employee', 'extras': {'signature': 'Co0CARFNMg/wlKkRlBYtvoxWyu1tF4mNbqRoloJBkPYrA/yLqovckS6AxfWVG66/t/HF8DiMFB1j8LNJCliPljC0b1YygxgeezxKIOllpV5imJYuRVetBLh1oCexTXnG++5FNHpW+oU1v+YbmTsaC4JOiL4n6r52Kf3uNFJDTOguv3eEYqAkdH6Qfpq/2y2MJSSSKiOKAdscdy4QAd+E6mAyY67qP13Obdo3RpewabPA/AUPX/tMURILXZjwJkPnFAwlXtml6WekoX2Fi+hTVtDlsKrvpe0rNaXEL9n/ajtzu96ewjiHXvM9wNuRiKIjvimZQ6IHlDIaGqVCz0OcZvfEuNegHSVEXkyt+zncSY8='}}]



Enter your query:  Show all tables in the database


...


Processing...

Answer:
[{'type': 'text', 'text': "The table 'Employee' already exists. Would you like to see its schema or perform other operations?", 'extras': {'signature': 'CtUBARFNMg9+7i/3mufEVPGf1o57a15iVGCe0tzL96JBv8ByYzNgOf+Jx/JICh04nbEaVp7lzlGuzn1oY7Ux2MV235PGxKCeGlBS+of0FawsBGLDI2iM560J8Jgg5tQiIoxszGx/xTTMbIXGs7LuqT736FGvIIjd1HW7cXqjjAkzebqXmQlwmP+fiCW+uRFLeIKfk5CF0T7fevQ4dcEtnjA83kOVxnpqBEjd7BWj3/tngf6vtHYXnLAkS3W0LtPN45dXOoFCd5vgTBR4iV8SoRYo1tTcH0rF'}}]



Enter your query:  Insert 3 employees: 1. Emp1, IT, 50000 2. Emp2, Finance, 45000 3. Emp3, Sales, 40000



Processing...

Answer:
[{'type': 'text', 'text': 'The 3 employees have been successfully inserted.', 'extras': {'signature': 'CvMBARFNMg9mk9XTAit08QBqqyI7abIHxoVXC6qP0mK+5Tkv/h93su3CUCxMtfXAMFnQNsQp6mEhD9PmeOTPecPGOVy7HNqXomzjHmneySVSSvy+/awlp4NMPsQjMCurILh2Exq4vK0n4CTuzsHDGchFCMX3AsHZ6mJrXhoA3vs4xxpOAn3muTScIxmSfmU68p67oS7DnveFZSLr0iS2fAyh69JTZK5ch2AlDAJ+T4saco7jA70MURbGUeik3ASuMrLsf2oOFWMDCU6FMr7w/FBF9BIDLJyE8QFxbX+mXS90+Q2gDgXm861tU7mXOiEAOrFnKw8r'}}]



Enter your query:  Show all employees



Processing...

Answer:
[{'type': 'text', 'text': 'Here are all the employees:\nID: 1, Name: Emp1, Department: IT, Salary: 50000.0\nID: 2, Name: Emp2, Department: Finance, Salary: 45000.0\nID: 3, Name: Emp3, Department: Sales, Salary: 40000.0', 'extras': {'signature': 'CrABARFNMg/4lEfSYNXodEAEet5BKxMP7YgbwOVktbQtRDW7U/zDUiG+YlAg+E1vq4RVrhQ7sBV8r68A6hZXey9pcZXjEpsmjzuyclSqXEjd3WBymEoDn9g1t6EMWETsNx0MiF6OYHvNB8C8RpkIeS4LWhdPYCoXXuMlxAIU/+3y0pNa5T6HECjrXp/h06W7G1oIM2ApTCJVS5awu2hro6SI83qgNMsZ07nF79o1vdBXh6o='}}]
